# NIT-4 · 2 — Allocating families to plans

**The situation.** The Office of Economic Opportunity has nine million dollars over
three years and a lattice of twenty plans it could offer: five guarantees by four tax
rates. Two instincts are on the table, and both are wrong in instructive ways.

The first is to **spread families evenly** across every plan, so that nothing is left
unmeasured. The second — the one an optimal-design textbook gives for a linear
response — is to **put everybody at the corners**, because that is where a straight
line is pinned down best. The corners of this lattice are a guarantee of 140 % of the
poverty line taxed at 80 %, and a family on that plan costs three times what a family
on the plan Congress is discussing costs. A design that ignores the invoice is a design
for a different budget.

The Conlisk–Watts allocation model — the piece of applied statistics the New Jersey
experiment is remembered for — was the answer to exactly this: information per dollar,
not per family. This notebook rebuilds that argument with `axiom.design`, and then does
the two things the model did not: it asks what each allocation *cannot* see, and it
looks at the calendar of four experiments before any of them runs.

| | reaches into |
|---|---|
| the candidate plans and what each costs | `surface.Design`, `W.plan_cost` |
| the planning surface and its information | `surface.Surface`, `design.fisher_information`, `design.expected_posterior_sd` |
| the allocation | `surface.optimal_exchange` |
| what a design cannot estimate | `design.identifiability_ridge`, `design.estimable_combinations` |
| how big the primary contrast can be seen | `design.mde`, `design.sample_size`, `design.power_from_se` |
| the randomization rule | `design.ArmAllocation`, `design.assign` |
| the calendar | `design.Occupancy`, `design.collisions`, `design.variance_inflation` |

In [ ]:
import sys

sys.path.insert(0, ".")

import numpy as np
import pandas as pd

import guarantee as W
from axiom.core import TimeWindow, is_failure
from axiom.design import (
    ArmAllocation, Assigned, Occupancy, Observation, assign, collisions, estimable_combinations,
    expected_posterior_sd, fisher_information, identifiability_ridge, mde, power_from_se,
    sample_size, variance_inflation,
)
from axiom.surface import PolynomialKernel, Surface, optimal_exchange
from axiom.display import enable, table

from _style import caption, compare, dumbbell, heat, intervals, lines, mark_x, mark_y, points

enable();  # every axiom result renders itself from here on

BUDGET = 9_000_000.0  # over the three enrolment years
YEARS = 3
NOISE_SD = 300.0  # sd of a family's three-year hours change, from the pilot interviews
lattice = W.lattice()
print(f"{lattice.n} candidate plans (including no plan), over {lattice.k} doses: {lattice.treatments}")

## 1 · The candidate set, and what each cell costs

Every plan on the lattice is a possible arm. What a family on it costs is the transfer
it would receive over three years, averaged over the earnings eligible families have,
plus the interviewing — and it runs from 300 USD a year for a control family to nearly
three thousand for the most generous corner.

In [ ]:
cost = np.array([[W.plan_cost(g, t) for t in W.TAX_RATES] for g in W.GUARANTEES])
fig = heat(
    cost, [f"t = {t}" for t in W.TAX_RATES], [f"G = {g}" for g in W.GUARANTEES],
    colorbar_title="USD / family-year", text_fmt="{:,.0f}",
    title="What a family on each plan costs, per year",
    subtitle=f"expected transfer plus {W.ADMIN_COST:.0f} USD of administration; a control family costs {W.ADMIN_COST:.0f}",
    x_title="tax rate", y_title="guarantee",
)
caption(fig, "The guarantee sets the cost and the tax rate claws some of it back. The top-left "
             "corner — a high guarantee at a low rate — is the plan that pays nearly every "
             "eligible family nearly everything, and it is the corner a D-optimal design for a "
             "linear surface most wants to sit in.")

## 2 · The planning surface

The response is modelled as two slopes and their product: hours change is linear in
the guarantee, linear in the tax rate, with an interaction. It is deliberately the
simplest surface anyone would defend, because the design has to be chosen before
there is data to defend a richer one — and section 5 asks what that choice hides.

For a surface linear in its parameters the Fisher information does not depend on
where the truth is, only on where the families are and how noisy their hours are.

In [ ]:
planning = Surface(W.planning_spec())
theta = {"alpha": 0.0, "beta1_guarantee": -80.0, "beta1_tax_rate": -110.0,
         "gamma_guarantee_tax_rate": -40.0, "sigma": NOISE_SD}
PRIOR_SDS = {"alpha": 100.0, "beta1_guarantee": 200.0, "beta1_tax_rate": 200.0,
             "gamma_guarantee_tax_rate": 100.0}
print("parameters of the mean:", [p.name for p in planning.model.parameters if p.name != "sigma"])
print("the surface is linear in all of them, so the design matrix is exact:")
lin = planning.linearize(W.design_data([(0.0, 0.0), W.NATIONAL_PLAN, (1.4, 0.8)]), theta)
print("  columns", lin.columns)
print("  rows   ", np.round(lin.X, 3).tolist())

## 3 · Three allocations, one budget

Each allocation is a set of shares over the lattice. Given the shares, the budget fixes
how many families it buys — and that number is the whole argument, because the
allocations differ in cost per family by a factor of two and a half.

- **spread**: forty per cent control, the rest evenly over the twenty plans — the instinct;
- **D-optimal**: Fedorov exchange on the planning surface, which puts every family at a corner;
- **two arms**: control against the plan on the table, and nothing else.

`optimal_exchange` returns proportions rather than a roster (a linear surface's optimal
design does not change shape with `n`), so the exchange runs at sixty rows and the
shares are then scaled to whatever the budget buys.

In [ ]:
exchange = optimal_exchange(planning, lattice, 60, [theta], criterion="d", seed=0)
assert not is_failure(exchange)
d_shares = pd.DataFrame(exchange.points, columns=["G", "t"]).value_counts(normalize=True).sort_index()
table([[f"G = {g}, t = {t}", f"{share:.0%}", f"{W.plan_cost(g, t):,.0f}"] for (g, t), share in d_shares.items()],
      headers=("plan", "share of families", "USD / family-year"),
      title=f"the D-optimal allocation (condition number {exchange.detail.get('condition', float('nan')):.1f})")


def n_for_budget(shares):
    per_family = sum(W.plan_cost(g, t) * s for (g, t), s in shares) * YEARS
    return int(BUDGET / per_family)


def roster(shares, n):
    return [pt for pt, s in shares for _ in range(int(round(s * n)))]


allocations = {
    "spread": [((0.0, 0.0), 0.40)] + [((g, t), 0.60 / 20) for g in W.GUARANTEES for t in W.TAX_RATES],
    "D-optimal": [((float(g), float(t)), float(s)) for (g, t), s in d_shares.items()],
    "two arms": [((0.0, 0.0), 0.5), (W.NATIONAL_PLAN, 0.5)],
}
rows = []
for name, shares in allocations.items():
    n = n_for_budget(shares)
    rows.append([name, n, f"{BUDGET / n / YEARS:,.0f}", int(round(shares[0][1] * n))])
table(rows, headers=("allocation", "families the budget buys", "USD per family-year", "of which control"))

Now the information each one buys. `fisher_information` at the budget-matched roster,
`expected_posterior_sd` under a wide prior for each parameter, and — the number the
decision actually turns on — the posterior sd of the **contrast at the national plan**,
which is a linear combination of the parameters and comes straight off the same matrix.

In [ ]:
def score(shares, surface=planning, prior=PRIOR_SDS, at=theta):
    pts = roster(shares, n_for_budget(shares))
    fi = fisher_information(surface, W.design_data(pts), at, noise_sd=NOISE_SD)
    sds = expected_posterior_sd(prior, fi)
    precision = np.diag([1.0 / prior[p] ** 2 if p in prior else 0.0 for p in fi.parameters])
    cov = np.linalg.inv(fi.as_array() + precision)
    ends = surface.linearize(W.design_data([(0.0, 0.0), W.NATIONAL_PLAN]), at)
    order = [list(ends.columns).index(p) for p in fi.parameters]
    c = (ends.X[1] - ends.X[0])[order]
    return fi, sds, float(np.sqrt(c @ cov @ c)), len(pts)


scores = {name: score(shares) for name, shares in allocations.items()}
table(
    [[name, n, f"{csd:.1f}", f"{sds['beta1_guarantee']:.0f}", f"{sds['beta1_tax_rate']:.0f}",
      f"{sds['gamma_guarantee_tax_rate']:.0f}", str(fi.singular)]
     for name, (fi, sds, csd, n) in scores.items()],
    headers=("allocation", "families", "sd of the national contrast", "sd of the G slope",
             "sd of the t slope", "sd of the interaction", "information singular"),
    title=f"the same {BUDGET / 1e6:.0f} million USD, three ways",
)

In [ ]:
fig = points(
    {name: ([csd], [sds["beta1_tax_rate"]]) for name, (fi, sds, csd, n) in scores.items()},
    size=14,
    title="What each allocation buys, and what it cannot",
    subtitle="posterior sd of the national contrast against that of the tax-rate slope, at equal budget",
    x_title=f"sd of the national contrast ({W.OUTCOME_UNIT})", y_title=f"sd of the tax-rate slope ({W.OUTCOME_UNIT})",
    height=420,
)
for name, (fi, sds, csd, n) in scores.items():
    fig.add_annotation(x=csd, y=sds["beta1_tax_rate"], text=name, showarrow=False, yshift=16,
                       font={"size": 11, "color": "#52514e"})
mark_y(fig, PRIOR_SDS["beta1_tax_rate"], text="the prior sd on a slope")
caption(fig, "Two arms pin the plan on the table best per dollar and learn nothing about the "
             "slopes — the tax-rate slope comes back close to its prior. The D-optimal corners "
             "learn the slopes best and pay for it in families. The spread is worse than both at "
             "what each is good at, and it is the only one of the three that can see a curve.")

## 4 · What the two-arm design cannot estimate

The two-arm information matrix is **singular**, and that is not a numerical accident.
Control against one plan is one contrast; the surface has three slope parameters.
`identifiability_ridge` names the flattest direction, and `estimable_combinations`
says how many independent combinations the design can estimate: one. The national
contrast is exactly that combination, so the two-arm experiment answers the question
on the table and no other — including the question Congress asks the week after,
when the tax rate on the table changes.

In [ ]:
slopes = ["beta1_guarantee", "beta1_tax_rate", "gamma_guarantee_tax_rate"]
two = roster(allocations["two arms"], n_for_budget(allocations["two arms"]))
ridge = identifiability_ridge(planning, W.design_data(two), theta, noise_sd=NOISE_SD, parameters=slopes)
assert not is_failure(ridge)
print("flattest direction:", {k: round(v, 2) for k, v in ridge.direction.items()},
      "| condition number:", ridge.condition_number)

report = estimable_combinations(
    [Observation(label="two_arm", expr=planning.expr, data=W.design_data(two[::10]), noise_sd=NOISE_SD)],
    theta, parameters=slopes, scaling="absolute", tolerance=1e-6, max_candidates=5000,
)
assert not is_failure(report)
print(f"rank {report.rank} of {len(report.parameters)}: {report.deficiency} directions the design cannot see")
for direction in report.null_basis:
    print("  flat:", {p: round(v, 2) for p, v in zip(report.parameters, direction)})

three = two + [(1.0, 0.5)] * (len(two) // 2)
report3 = estimable_combinations(
    [Observation(label="three_arm", expr=planning.expr, data=W.design_data(three[::10]), noise_sd=NOISE_SD)],
    theta, parameters=slopes, scaling="absolute", tolerance=1e-6, max_candidates=5000,
)
print(f"\nadd a third arm at G = 1.0: rank {report3.rank} of {len(report3.parameters)}")

## 5 · What the corners cannot see

Now suppose the planning surface is wrong in the direction the world actually is: the
response to the tax rate has a curvature. Give the planning surface a quadratic term
in `t` and ask each allocation how well it would estimate the new coefficient. The
corner design has exactly two tax rates in it, and two points do not determine a
parabola.

In [ ]:
curved = Surface(W.spec(
    {"guarantee": PolynomialKernel(reference_dose=max(W.GUARANTEES), amplitude_scale=200.0, degree=1),
     "tax_rate": PolynomialKernel(reference_dose=max(W.TAX_RATES), amplitude_scale=200.0, degree=2)},
    name="nit4_curved", unit_labels=("planning",),
))
theta_curved = {**theta, "beta2_tax_rate": -25.0}
prior_curved = {**PRIOR_SDS, "beta2_tax_rate": 200.0}
rows, labels, before, after = [], [], [], []
for name, shares in allocations.items():
    fi, sds, csd, n = score(shares, surface=curved, prior=prior_curved, at=theta_curved)
    rows.append([name, str(fi.singular), f"{sds['beta2_tax_rate']:.0f}", f"{csd:.1f}"])
    labels.append(name); before.append(prior_curved["beta2_tax_rate"]); after.append(sds["beta2_tax_rate"])
table(rows, headers=("allocation", "information singular", "sd of the curvature", "sd of the national contrast"),
      title="the same three allocations, asked about a curve")
fig = dumbbell(
    labels, before, after, before_label="prior", after_label="posterior",
    title="Only the spread can see a curve",
    subtitle="posterior sd of the tax-rate curvature under each allocation, against its prior of 200",
    x_title=f"sd of the curvature coefficient ({W.OUTCOME_UNIT})",
)
caption(fig, "The corner design and the two-arm design hand the curvature's prior straight back "
             "— the information matrix is singular in that direction, and no number of families "
             "changes it. A design chosen against the model you hope is right cannot tell you "
             "whether it is. That is the argument for interior points, and it is the argument "
             "the four sites actually followed.")

## 6 · The allocation the programme takes

Not the spread and not the corners. Each site takes an interior set of plans around
the ones Congress is discussing, weighted toward the cheaper cells, with forty per cent
of families in control — which is what the four experiments did, and what
`guarantee.SITES` records. Score that programme as one design.

In [ ]:
programme_shares = []
n_total = sum(s.n_families for s in W.SITES)
for site in W.SITES:
    weight = site.n_families / n_total
    for arm, share in zip(site.arms, site.shares):
        programme_shares.append((site.plan_of(arm), weight * share))
fi_p, sds_p, csd_p, n_p = score(programme_shares)
programme_cost = sum(W.plan_cost(g, t) * s for (g, t), s in programme_shares) * YEARS * n_total
print(f"{n_total:,} families over four sites, {len({p for p, _ in programme_shares})} distinct plans, "
      f"{programme_cost / 1e6:.1f} million USD over three years")
table([["the four sites as one design", n_total, f"{score(programme_shares)[2] * np.sqrt(n_p / n_total):.1f}",
        f"{sds_p['beta1_guarantee'] * np.sqrt(n_p / n_total):.0f}", f"{sds_p['beta1_tax_rate'] * np.sqrt(n_p / n_total):.0f}",
        f"{sds_p['gamma_guarantee_tax_rate'] * np.sqrt(n_p / n_total):.0f}"]],
      headers=("design", "families", "sd of the national contrast", "sd of the G slope", "sd of the t slope", "sd of the interaction"),
      title="scaled to the families the sites actually enrolled")

m = mde(n_total, NOISE_SD)
n_needed = sample_size(W.HOURS_THRESHOLD / 3.0, NOISE_SD)
print(f"\nminimum detectable two-arm contrast with {n_total:,} families: {m.effect:.0f} {W.OUTCOME_UNIT}")
print(f"families needed to detect a third of the Treasury's threshold at 80 % power: {n_needed.n:,}")
print(f"power to see the threshold itself at the programme's contrast sd: "
      f"{power_from_se(W.HOURS_THRESHOLD, csd_p * np.sqrt(n_p / n_total)).power:.3f}")

## 7 · The randomization rule

Families are randomized within income band, in permuted blocks, at each site. The rule
is an `ArmAssignment` — allocation, method, seed, the hash of the strata and of the
roster — and it is what gets filed with the experiment in notebook 3, because six years
later somebody will ask who was in which arm and the rule re-derives it. The roster
does not go into the rule; `verify()` puts the two back together.

In [ ]:
site = W.SITE_BY_NAME["new_jersey"]
families = W.roster(site, seed=W.SEED)
split: ArmAllocation = W.allocation(site)
assigned: Assigned = W.assign_site(site, families, seed=W.SEED + 1)
print(f"{site.experiment}: {len(families)} families into {split.n_arms} arms, block size {split.block_size()}")
print("counts:", assigned.counts())
table([[r.covariate, f"{r.means[0]:,.0f}", f"{np.nanmean(r.means[1:]):,.0f}", f"{r.smd:+.3f}"]
       for r in assigned.spec.balance],
      headers=("covariate", "control mean", "mean over plan arms", "worst standardized gap"),
      title=f"balance after blocking within income band (worst gap between any two of nine arms: smd {assigned.spec.worst_smd:.3f})")
print("\nthe rule re-derives the arms:", assigned.verify().status)
print("rule hash", assigned.spec.content_hash()[:12], "| roster hash", assigned.spec.roster_hash[:12],
      "| the roster is in the rule:", families.family.iloc[0] in assigned.spec.to_json())

## 8 · The calendar

Four experiments in four sets of cities over fourteen years, and inside one of them a
second treatment — manpower counselling and training subsidies — randomized to the
same families in the same years. `collisions` reads the calendar: which pairs share
units and periods, and whether they move the same lever.

In [ ]:
calendar = [
    Occupancy(experiment="NJ-1968", units=("trenton", "paterson", "jersey_city", "scranton"),
              window=TimeWindow(start=0, stop=4), treatments=("plan",)),
    Occupancy(experiment="RURAL-1970", units=("iowa", "north_carolina"),
              window=TimeWindow(start=2, stop=5), treatments=("plan",)),
    Occupancy(experiment="GARY-1971", units=("gary",),
              window=TimeWindow(start=3, stop=7), treatments=("plan",)),
    Occupancy(experiment="SIME-DIME-1971", units=("seattle", "denver"),
              window=TimeWindow(start=3, stop=14), treatments=("plan",)),
    Occupancy(experiment="SIME-TRAINING", units=("seattle", "denver"),
              window=TimeWindow(start=3, stop=8), treatments=("training",)),
]
found = collisions(calendar)
table([[c.left, c.right, c.kind, ", ".join(c.units), len(c.periods), ", ".join(c.treatments) or "—", c.verdict().status]
       for c in found],
      headers=("left", "right", "kind", "shared units", "years", "shared lever", "verdict"),
      title=f"{len(found)} overlapping pair(s) of {len(collisions(calendar, include_disjoint=True))} on the calendar")
inflation = variance_inflation(abs(W.TRUTH.training_effect), 0.5, NOISE_SD)
print(f"\nwhat the training programme costs the income-maintenance readout: a variance factor of "
      f"{inflation:.4f}, if its effect is {abs(W.TRUTH.training_effect):.0f} hours on half the families")
print("what it costs if the two treatments interact: not a variance factor — notebook 3 checks")

One pair overlaps, and it is `concurrent`, not `confounded`: two independently
randomized treatments in the same families and years move *different* levers, so
neither estimate is biased by the other. The overlap costs variance — and here almost
none of it, because the training effect is small against the noise — plus one
assumption: that the two treatments do not interact. That assumption's `challenged_by`
names a factorial analysis, and notebook 3 runs it.

## What notebook 2 decided

1. **Allocation is a budget problem.** The same nine million dollars buys about
   3,550 families in two arms, 2,090 across the lattice, or 1,440 at the D-optimal
   corners, because a family at a corner costs two and a half times a family on the
   plan on the table.
2. **Two arms answer one question.** They pin the national contrast to 10 hours —
   the best of the three per dollar — and their information matrix is singular in
   two of the three slope directions. `estimable_combinations` says rank one: the
   contrast itself, and nothing that would survive a change of tax rate.
3. **The corners learn the slopes and cannot see a curve.** A D-optimal design for a
   linear surface has two tax rates in it. Asked about a curvature it hands the
   prior back, singular, however many families it has.
4. **The programme's interior design is the compromise the sites made**, and it is a
   good one: with the families the four sites enrolled it sees the national contrast
   to about seven hours and the threshold itself with power indistinguishable from one.
5. **The rule travels, the roster does not.** Blocked within income band, the
   assignment re-derives from its own record, and that record is what notebook 3 files.
6. **The calendar has one overlap and it is a cost, not a conflict.** The training
   programme inside Seattle-Denver is concurrent; what it leaves open is an
   interaction, which is a check rather than an assumption.